# PRISM OOD - CONCH (Embedding-Based, Fast)
## Uses pre-saved embeddings - no GPU needed!
### 4 pairs: PCam↔MHIST | CRC↔BRACS (binarized)

In [1]:
import os, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss
from scipy.optimize import minimize_scalar
from google.colab import drive
drive.mount('/content/drive')

EMB_DIR     = '/content/drive/MyDrive/PRISM/embeddings/conch'
RESULTS_DIR = '/content/drive/MyDrive/PRISM/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

LABEL_FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
SEEDS = [42, 123, 456]
MKEY = 'conch'

# CRC class order (sorted folder names in NCT-CRC-HE-100K):
# ADI=0, BACK=1, DEB=2, LYM=3, MUC=4, MUS=5, NORM=6, STR=7, TUM=8
CRC_BINARY = lambda y: (y == 8).astype(int)   # TUM=1, rest=0

# BRACS class order (sorted folder names):
# ADH=0, DCIS=1, FEA=2, IC=3, N=4, PB=5, UDH=6
BRACS_BINARY = lambda y: np.isin(y, [1, 3]).astype(int)  # DCIS=1, IC=1, rest=0

print(f"Model: CONCH")
print(f"Embedding dir: {EMB_DIR}")
print("No GPU needed - using pre-saved embeddings!")

Mounted at /content/drive
Model: CONCH
Embedding dir: /content/drive/MyDrive/PRISM/embeddings/conch
No GPU needed - using pre-saved embeddings!


In [2]:
# Load all embeddings from Drive
def load_emb(dataset, split):
    path = f'{EMB_DIR}/{dataset}'
    feats  = np.load(f'{path}/{split}_features.npy')
    labels = np.load(f'{path}/{split}_labels.npy')
    return feats, labels

print("Loading embeddings...")

# Binary datasets (already binary labels: 0/1)
pcam_train_X,  pcam_train_y  = load_emb('pcam',  'train')
pcam_test_X,   pcam_test_y   = load_emb('pcam',  'test')
mhist_train_X, mhist_train_y = load_emb('mhist', 'train')
mhist_test_X,  mhist_test_y  = load_emb('mhist', 'test')

# Multiclass datasets -> binarize labels
crc_train_X,   _crc_train_y  = load_emb('crc',   'train')
crc_test_X,    _crc_test_y   = load_emb('crc',   'test')
crc_train_y  = CRC_BINARY(_crc_train_y)
crc_test_y   = CRC_BINARY(_crc_test_y)

bracs_train_X, _bracs_train_y = load_emb('bracs', 'train')
bracs_test_X,  _bracs_test_y  = load_emb('bracs', 'test')
bracs_train_y = BRACS_BINARY(_bracs_train_y)
bracs_test_y  = BRACS_BINARY(_bracs_test_y)

print(f"PCam train: {pcam_train_X.shape} | test: {pcam_test_X.shape}")
print(f"MHIST train: {mhist_train_X.shape} | test: {mhist_test_X.shape}")
print(f"CRC binary train: {crc_train_X.shape} dist={np.bincount(crc_train_y)}")
print(f"BRACS binary train: {bracs_train_X.shape} dist={np.bincount(bracs_train_y)}")

Loading embeddings...
PCam train: (262144, 512) | test: (32768, 512)
MHIST train: (1849, 512) | test: (977, 512)
CRC binary train: (70000, 512) dist=[59882 10118]
BRACS binary train: (3657, 512) dist=[2319 1338]


In [3]:
def compute_ece(probs, labels, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (probs >= bins[i]) & (probs < bins[i+1])
        if mask.sum() > 0:
            ece += mask.mean() * abs(labels[mask].mean() - probs[mask].mean())
    return float(ece)

def temperature_scale_binary(logits, labels):
    def nll(T):
        p = np.clip(1 / (1 + np.exp(-logits / T)), 1e-7, 1-1e-7)
        return -np.mean(labels * np.log(p) + (1-labels) * np.log(1-p))
    return minimize_scalar(nll, bounds=(0.1, 10.0), method='bounded').x

def stratified_sample(n_total, labels, fraction, seed):
    np.random.seed(seed)
    indices = np.arange(n_total)
    classes = np.unique(labels)
    sampled = []
    for c in classes:
        c_idx = indices[labels == c]
        n = max(1, int(len(c_idx) * fraction))
        sampled.extend(np.random.choice(c_idx, size=n, replace=False))
    return np.array(sampled)

print("Helper functions ready!")

Helper functions ready!


In [4]:
def run_ood(src_X, src_y, src_name, tgt_X, tgt_y, tgt_name):
    print(f"\n{'='*55}")
    print(f"OOD: {src_name} -> {tgt_name}")
    print(f"Src: {src_X.shape} | Tgt: {tgt_X.shape}")
    print(f"Src dist: {np.bincount(src_y.astype(int))} | Tgt dist: {np.bincount(tgt_y.astype(int))}")
    print(f"{'='*55}")

    results, ts_results = [], []

    for frac in LABEL_FRACTIONS:
        for seed in SEEDS:
            sampled = stratified_sample(len(src_y), src_y.astype(int), frac, seed)
            X_tr, y_tr = src_X[sampled], src_y[sampled].astype(int)

            clf = LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
            clf.fit(X_tr, y_tr)

            logits = clf.decision_function(tgt_X)
            probs  = 1 / (1 + np.exp(-logits))
            preds  = (probs > 0.5).astype(int)
            tgt_int = tgt_y.astype(int)

            try:    auroc = roc_auc_score(tgt_int, probs)
            except: auroc = float('nan')
            ece_raw = compute_ece(probs, tgt_int)
            brier   = float(brier_score_loss(tgt_int, probs))
            f1      = float(f1_score(tgt_int, preds, average='macro', zero_division=0))

            try:
                T      = temperature_scale_binary(logits, tgt_int)
                sp     = 1 / (1 + np.exp(-logits / T))
                ece_sc = compute_ece(sp, tgt_int)
                ece_imp = ece_raw - ece_sc
            except:
                T = ece_sc = ece_imp = float('nan')

            results.append({'fraction':frac,'seed':seed,'auroc':auroc,
                            'ece':ece_raw,'f1':f1,'brier':brier})
            ts_results.append({'fraction':frac,'seed':seed,'temperature':T,
                               'ece_raw':ece_raw,'ece_scaled':ece_sc,'ece_improvement':ece_imp})
            print(f"  {frac:.0%} s{seed}: AUROC={auroc:.4f} ECE={ece_raw:.4f} F1={f1:.4f}")

    tag = f"{MKEY}_ood_{src_name.lower()}_to_{tgt_name.lower()}"
    pd.DataFrame(results).to_csv(f'{RESULTS_DIR}/{tag}_results.csv', index=False)
    pd.DataFrame(ts_results).to_csv(f'{RESULTS_DIR}/{tag}_temperature_scaling.csv', index=False)
    print(f"✅ Saved: {tag}")
    return pd.DataFrame(results)

print("run_ood() ready!")

run_ood() ready!


In [5]:
# Run all 4 OOD pairs - fast, no GPU!
r1 = run_ood(pcam_train_X,  pcam_train_y,  'PCam',  mhist_test_X,  mhist_test_y,  'MHIST')
r2 = run_ood(mhist_train_X, mhist_train_y, 'MHIST', pcam_test_X,   pcam_test_y,   'PCam')
r3 = run_ood(crc_train_X,   crc_train_y,   'CRC',   bracs_test_X,  bracs_test_y,  'BRACS')
r4 = run_ood(bracs_train_X, bracs_train_y, 'BRACS', crc_test_X,    crc_test_y,    'CRC')

print("\n🎉 ALL 4 OOD PAIRS DONE!")


OOD: PCam -> MHIST
Src: (262144, 512) | Tgt: (977, 512)
Src dist: [131072 131072] | Tgt dist: [617 360]
  1% s42: AUROC=0.6396 ECE=0.0920 F1=0.5459
  1% s123: AUROC=0.6679 ECE=0.0862 F1=0.5480
  1% s456: AUROC=0.6601 ECE=0.1480 F1=0.4681
  5% s42: AUROC=0.6425 ECE=0.1042 F1=0.5892
  5% s123: AUROC=0.6482 ECE=0.1347 F1=0.5489
  5% s456: AUROC=0.6431 ECE=0.1187 F1=0.5653
  10% s42: AUROC=0.6372 ECE=0.1289 F1=0.5906
  10% s123: AUROC=0.6410 ECE=0.1423 F1=0.5686
  10% s456: AUROC=0.6294 ECE=0.1350 F1=0.5615
  25% s42: AUROC=0.6360 ECE=0.1433 F1=0.5889
  25% s123: AUROC=0.6240 ECE=0.1437 F1=0.5790
  25% s456: AUROC=0.6350 ECE=0.1380 F1=0.5889
  50% s42: AUROC=0.6260 ECE=0.1364 F1=0.5848
  50% s123: AUROC=0.6206 ECE=0.1615 F1=0.5630
  50% s456: AUROC=0.6308 ECE=0.1424 F1=0.5702
  100% s42: AUROC=0.6332 ECE=0.1507 F1=0.5727
  100% s123: AUROC=0.6334 ECE=0.1506 F1=0.5735
  100% s456: AUROC=0.6334 ECE=0.1506 F1=0.5735
✅ Saved: conch_ood_pcam_to_mhist

OOD: MHIST -> PCam
Src: (1849, 512) | Tgt:

In [6]:
for name, r in [('PCam→MHIST',r1),('MHIST→PCam',r2),('CRC→BRACS',r3),('BRACS→CRC',r4)]:
    s = r.groupby('fraction')[['auroc','ece','f1']].mean()
    print(f"\n--- {name} (mean over 3 seeds) ---")
    print(s.round(4).to_string())


--- PCam→MHIST (mean over 3 seeds) ---
           auroc     ece      f1
fraction                        
0.01      0.6559  0.1087  0.5206
0.05      0.6446  0.1192  0.5678
0.10      0.6359  0.1354  0.5736
0.25      0.6317  0.1416  0.5856
0.50      0.6258  0.1468  0.5727
1.00      0.6334  0.1506  0.5732

--- MHIST→PCam (mean over 3 seeds) ---
           auroc     ece      f1
fraction                        
0.01      0.6179  0.2247  0.3334
0.05      0.7301  0.2694  0.3334
0.10      0.7541  0.3219  0.3334
0.25      0.7186  0.3981  0.3334
0.50      0.7036  0.4388  0.3335
1.00      0.6955  0.4550  0.3342

--- CRC→BRACS (mean over 3 seeds) ---
           auroc     ece      f1
fraction                        
0.01      0.4105  0.2362  0.4232
0.05      0.4130  0.2794  0.4196
0.10      0.4253  0.2992  0.4368
0.25      0.4272  0.3175  0.4430
0.50      0.4310  0.3237  0.4423
1.00      0.4285  0.3347  0.4407

--- BRACS→CRC (mean over 3 seeds) ---
           auroc     ece      f1
fraction         